# Integrating the variant table with sample metadata

This notebook shows **data engineering steps only**: loading the pipeline's variant table,
harmonizing a deliberately messy metadata sheet, and joining the two into one tidy table.

> **Demo disclaimer.** The metadata are synthetic, there is a single real sample, and the
> genomic window is 500 kb. Nothing here supports any biological or clinical interpretation.
> Counts below are shown only to check that the join and the formatting worked.

**Inputs**
- `results/tables/HG001.variants.tsv`, produced by `snakemake` (`scripts/vep_to_table.py`)
- `metadata/example_sample_metadata.tsv`, a synthetic metadata sheet

**Output**
- `results/integrated/variants_with_metadata.tsv`

In [1]:
from pathlib import Path
import re

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VARIANTS_TSV = ROOT / "results" / "tables" / "HG001.variants.tsv"
METADATA_TSV = ROOT / "metadata" / "example_sample_metadata.tsv"
OUT_TSV = ROOT / "results" / "integrated" / "variants_with_metadata.tsv"

if not VARIANTS_TSV.exists():
    raise FileNotFoundError(f"{VARIANTS_TSV} not found - run `snakemake --cores 4` first.")
pd.set_option("display.max_columns", 30)

## 1. Load the variant table

One row per *(variant, ALT allele, VEP transcript consequence)*. Every column is read as
text first and only known numeric columns are converted. This stops pandas from guessing
types, for example turning the `pick` flag into `1.0` or reading `exon` = `3/12` as a date.

In [2]:
numeric_dtypes = {
    "pos": "Int64", "qual": "Float64", "dp": "Int64", "gq": "Int64",
    "ad_ref": "Int64", "ad_alt": "Int64",
}
variants = (pd.read_csv(VARIANTS_TSV, sep="\t", dtype="string")
            .astype(numeric_dtypes))
print(variants.shape)
variants.head()

(7308, 25)


,sample_id,chrom,pos,ref,alt,qual,filter,gt,dp,gq,ad_ref,ad_alt,consequence,impact,symbol,gene,feature_type,feature,biotype,exon,intron,hgvsc,hgvsp,variant_class,pick
0,HG001,20,33000521,G,A,428.64,PASS,0/1,22,99,8,14,intron_variant,MODIFIER,SUN5,ENSG00000167098,Transcript,ENST00000356173,protein_coding,<NA>,4/12,ENST00000356173.8:c.279-386C>T,<NA>,SNV,1
1,HG001,20,33000521,G,A,428.64,PASS,0/1,22,99,8,14,intron_variant,MODIFIER,SUN5,ENSG00000167098,Transcript,ENST00000375519,protein_coding,<NA>,3/3,ENST00000375519.3:c.212-386C>T,<NA>,SNV,<NA>
2,HG001,20,33000521,G,A,428.64,PASS,0/1,22,99,8,14,intron_variant,MODIFIER,SUN5,ENSG00000167098,Transcript,ENST00000375523,protein_coding,<NA>,3/11,ENST00000375523.8:c.204-386C>T,<NA>,SNV,<NA>
3,HG001,20,33000521,G,A,428.64,PASS,0/1,22,99,8,14,intron_variant,MODIFIER,SUN5,ENSG00000167098,Transcript,ENST00000420875,protein_coding,<NA>,4/4,ENST00000420875.5:c.246-386C>T,<NA>,SNV,<NA>
4,HG001,20,33000521,G,A,428.64,PASS,0/1,22,99,8,14,intron_variant,MODIFIER,SUN5,ENSG00000167098,Transcript,ENST00001096147,protein_coding,<NA>,3/3,ENST00001096147.1:c.*31-386C>T,<NA>,SNV,<NA>


## 2. Harmonize the variant table

- Build a stable `variant_id` (`chrom-pos-ref-alt`) and a UCSC-style chromosome name for
  joining with tools that use the `chr` prefix.
- Keep one consequence per ALT allele (`pick == "1"`, set by VEP `--flag_pick_allele`) for a
  variant-level view, while the full table stays available.
- Split multi-term consequences (`missense_variant&splice_region_variant`) and keep the
  first term as `primary_consequence`.
- Derive `zygosity` from the genotype string (handles phased `|` and unphased `/`).

In [3]:
v = variants.copy()
v["variant_id"] = v["chrom"] + "-" + v["pos"].astype("string") + "-" + v["ref"] + "-" + v["alt"]
v["chrom_ucsc"] = "chr" + v["chrom"].str.removeprefix("chr")
v["primary_consequence"] = v["consequence"].str.split("&").str[0].astype("string")
v["filter_pass"] = v["filter"].eq("PASS")


def zygosity(gt):
    if pd.isna(gt):
        return pd.NA
    alleles = re.split(r"[/|]", gt)
    if any(a == "." for a in alleles):
        return "missing"
    return "hom" if len(set(alleles)) == 1 else "het"


v["zygosity"] = v["gt"].map(zygosity).astype("string")
v["vaf"] = (v["ad_alt"] / (v["ad_ref"] + v["ad_alt"])).round(3)

variant_level = v[v["pick"].eq("1")].drop_duplicates("variant_id")
print(f"{len(v):,} consequence rows -> {len(variant_level):,} variant-level rows")
variant_level[["variant_id", "gt", "zygosity", "dp", "vaf", "filter",
               "primary_consequence", "impact", "symbol"]].head()

7,308 consequence rows -> 530 variant-level rows


,variant_id,gt,zygosity,dp,vaf,filter,primary_consequence,impact,symbol
0,20-33000521-G-A,0/1,het,22,0.636,PASS,intron_variant,MODIFIER,SUN5
7,20-33001519-C-CTTTCTTTCTTTCTTTCTTTCTTTCT,1/1,hom,25,1.0,PASS,intron_variant,MODIFIER,SUN5
14,20-33001641-T-TCCTCCCTC,1/1,hom,6,1.0,PASS,intron_variant,MODIFIER,SUN5
21,20-33001755-A-G,1/1,hom,19,1.0,PASS,intron_variant,MODIFIER,SUN5
28,20-33002172-G-GA,1/1,hom,20,1.0,PASS,intron_variant,MODIFIER,SUN5


## 3. Load and harmonize the metadata

The sheet mimics common real-world problems: inconsistent column names, sample IDs with
different case/separators and trailing spaces, mixed date formats, free-text categories,
and quantities with mixed units.

In [4]:
meta_raw = pd.read_csv(METADATA_TSV, sep="\t", comment="#", dtype="string")
meta_raw

,Sample ID,Seq Platform,library_prep,Collection Date,DNA input,Batch,Notes
0,hg001,illumina hiseq 2500,TruSeq PCR-free,2014/01/15,50 ng,B1,demo row linked to pipeline output
1,DEMO-002,Illumina,truseq pcr free,15-02-2014,0.1 ug,b1,"fictitious sample, no sequencing data"
2,demo_003,ILLUMINA NovaSeq,Nextera Flex,2014-03-01,<NA>,B2,"fictitious sample, no sequencing data"


In [5]:
def snake_case(name: str) -> str:
    return re.sub(r"[^0-9a-z]+", "_", name.strip().lower()).strip("_")


def normalize_sample_id(x):
    # 'hg001 ', 'DEMO-002', 'demo_003' -> 'HG001', 'DEMO002', 'DEMO003'
    return pd.NA if pd.isna(x) else re.sub(r"[^0-9A-Z]", "", x.upper())


def parse_date(x):
    # Accept the formats present in the sheet; fail loudly on anything else.
    if pd.isna(x):
        return pd.NaT
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    raise ValueError(f"Unrecognised date: {x!r}")


UNIT_TO_NG = {"ng": 1.0, "ug": 1_000.0, "µg": 1_000.0}


def to_ng(x):
    if pd.isna(x):
        return pd.NA
    m = re.fullmatch(r"\s*([0-9.]+)\s*(ng|ug|µg)\s*", x, flags=re.IGNORECASE)
    if not m:
        raise ValueError(f"Unrecognised quantity: {x!r}")
    return float(m.group(1)) * UNIT_TO_NG[m.group(2).lower()]


PLATFORM_MAP = {"illumina": "ILLUMINA"}  # vendor only; model kept separately


meta = meta_raw.rename(columns=snake_case)
meta = meta.assign(
    sample_id=meta["sample_id"].map(normalize_sample_id).astype("string"),
    platform=meta["seq_platform"].str.split().str[0].str.lower().map(PLATFORM_MAP).astype("string"),
    instrument_model=meta["seq_platform"].str.split(n=1).str[1].str.strip().str.upper().astype("string"),
    library_prep=meta["library_prep"].str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).astype("string"),
    collection_date=meta["collection_date"].map(parse_date),
    dna_input_ng=meta["dna_input"].map(to_ng).astype("Float64"),
    batch=meta["batch"].str.upper().astype("string"),
).drop(columns=["seq_platform", "dna_input"])

assert meta["sample_id"].is_unique, "duplicate sample IDs after normalization"
meta

,sample_id,library_prep,collection_date,batch,notes,platform,instrument_model,dna_input_ng
0,HG001,truseq_pcr_free,2014-01-15,B1,demo row linked to pipeline output,ILLUMINA,HISEQ 2500,50.0
1,DEMO002,truseq_pcr_free,2014-02-15,B1,"fictitious sample, no sequencing data",ILLUMINA,<NA>,100.0
2,DEMO003,nextera_flex,2014-03-01,B2,"fictitious sample, no sequencing data",ILLUMINA,NOVASEQ,<NA>


## 4. Join variants with metadata

`validate="many_to_one"` makes pandas raise an error if the metadata ever contain duplicated
sample IDs, and `indicator=True` records where each row came from, so samples missing on
either side are reported instead of silently dropped.

In [6]:
v_keyed = v.assign(sample_id=v["sample_id"].map(normalize_sample_id).astype("string"))

merged = v_keyed.merge(meta, on="sample_id", how="outer", validate="many_to_one", indicator=True)

print(merged["_merge"].value_counts().to_string())
print("\nSamples in metadata without variant data:",
      sorted(merged.loc[merged["_merge"] == "right_only", "sample_id"].dropna().unique()))

_merge
both          7308
right_only       2
left_only        0

Samples in metadata without variant data: ['DEMO002', 'DEMO003']


In [7]:
integrated = merged[merged["_merge"] == "both"].drop(columns="_merge")

front = ["sample_id", "variant_id", "chrom", "chrom_ucsc", "pos", "ref", "alt", "gt", "zygosity",
         "dp", "gq", "vaf", "filter", "filter_pass", "primary_consequence", "impact", "symbol",
         "feature", "pick", "platform", "instrument_model", "library_prep", "batch",
         "collection_date", "dna_input_ng"]
integrated = integrated[front + [c for c in integrated.columns if c not in front]]

OUT_TSV.parent.mkdir(parents=True, exist_ok=True)
integrated.to_csv(OUT_TSV, sep="\t", index=False, date_format="%Y-%m-%d")
print(f"wrote {OUT_TSV.relative_to(ROOT)}  ({len(integrated):,} rows x {integrated.shape[1]} cols)")

wrote results/integrated/variants_with_metadata.tsv  (7,308 rows x 38 cols)


## 5. Sanity checks on the integrated table

These are **format checks** (completeness, types, category levels), not results.

In [8]:
checks = pd.DataFrame({
    "dtype": integrated.dtypes.astype(str),
    "missing_pct": (integrated.isna().mean() * 100).round(1),
    "n_unique": integrated.nunique(),
}).loc[front]
checks

,dtype,missing_pct,n_unique
sample_id,string,0.0,1
variant_id,string,0.0,530
chrom,string,0.0,1
chrom_ucsc,string,0.0,1
pos,Int64,0.0,522
ref,string,0.0,36
alt,string,0.0,46
gt,string,0.0,3
zygosity,string,0.0,2
dp,Int64,0.0,42


In [9]:
variant_level_int = integrated[integrated["pick"].eq("1")].drop_duplicates("variant_id")
(variant_level_int
 .groupby(["filter_pass", "zygosity"], dropna=False)
 .size()
 .rename("n_variants")
 .reset_index())

,filter_pass,zygosity,n_variants
0,False,het,2
1,True,het,330
2,True,hom,198


## Summary

- The variant table was typed explicitly and given stable keys (`variant_id`, `chrom_ucsc`).
- The metadata were cleaned up: snake_case columns, canonical sample IDs, ISO dates,
  controlled vocabularies, and DNA amounts converted to ng.
- The join was validated for cardinality and reports unmatched samples.

The same steps scale to many samples. Add rows to `config/samples.tsv` and to the
metadata sheet, then rerun the pipeline and this notebook.